# 02회차 실습 노트북 — 핀테크 데이터 수집 기술

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adminmrobo/fintech_data_analysis/blob/main/notebooks/02_data_collection.ipynb)

해설 페이지: [https://adminmrobo.github.io/fintech_data_analysis/02_data_collection.html](https://adminmrobo.github.io/fintech_data_analysis/02_data_collection.html)

위에서부터 셀을 차례로 실행하세요(Shift+Enter). 데이터는 저장소 `data/` 폴더(fintech-data 복사본)에서 읽으며, 무작위 시드를 42로 고정해 해설 페이지의 대결 숫자와 같은 값이 나옵니다.

글 · 안상선 ((주)M-Robo 대표)

In [1]:
# 준비: 라이브러리와 데이터 불러오기 (Colab에서는 그대로 실행하면 됩니다)
import os, urllib.parse, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)

BASE = 'https://raw.githubusercontent.com/adminmrobo/fintech_data_analysis/main/data/'
def load(path, **kw):
    """저장소의 data/ 폴더에서 CSV를 읽는다. 내 컴퓨터에 저장소가 있으면 그 파일을, 없으면 GitHub에서 읽는다."""
    for root in ('data', '../data'):
        f = os.path.join(root, path)
        if os.path.exists(f):
            return pd.read_csv(f, **kw)
    return pd.read_csv(BASE + urllib.parse.quote(path), **kw)

RS = 42   # 해설 페이지와 같은 숫자가 나오도록 무작위 시드를 고정

## 1. 수집한 데이터 두 가지 살펴보기

In [2]:
ins = load('Revolution/insurance.csv'); ph = load('Revolution2/Website Phishing.csv')
print(ins.shape, ph.shape)
print(ph.Result.value_counts())   # 원자료: 1 = 피싱, 0 = 의심, -1 = 정상
ins.head()

(1338, 7) (1353, 10)
Result
-1    702
 1    548
 0    103
Name: count, dtype: int64


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


## 2. 대결 재현: 점검한 600건 vs 라벨 오류가 섞인 전부
깨끗한 라벨 600건으로 학습한 모델과, 학습 세트 전부를 쓰되 라벨의 20%를 일부러 뒤집은 모델을 비교합니다. 이어서 라벨이 **한쪽으로** 틀린 경우(피싱을 정상으로 40% 잘못 적음)도 봅니다.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score

X = ph.drop(columns='Result').values; y = (ph.Result == 1).astype(int).values
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.3, random_state=RS, stratify=y)
rng = np.random.default_rng(RS)
idx = rng.choice(len(ytr), size=min(1000, len(ytr)), replace=False)
noisy = ytr.copy(); flip = rng.random(len(noisy)) < .2; noisy[flip] = 1 - noisy[flip]

clean = LogisticRegression(max_iter=2000).fit(Xtr[idx[:600]], ytr[idx[:600]])
rand = LogisticRegression(max_iter=2000).fit(Xtr, noisy)
biased_y = ytr.copy(); f2 = (ytr == 1) & (rng.random(len(ytr)) < .4); biased_y[f2] = 0
biased = LogisticRegression(max_iter=2000).fit(Xtr, biased_y)
for name, m in [('깨끗한 600건', clean), ('전부 + 무작위 오류 20%', rand), ('전부 + 한쪽 오류', biased)]:
    print(f'{name:16s} 정확도 {m.score(Xte, yte):.3f}  피싱 재현율 {recall_score(yte, m.predict(Xte)):.3f}')

깨끗한 600건         정확도 0.894  피싱 재현율 0.902
전부 + 무작위 오류 20%  정확도 0.877  피싱 재현율 0.927
전부 + 한쪽 오류       정확도 0.788  피싱 재현율 0.524


**AI 판단 연습** — 무작위 오류와 한쪽으로 치우친 오류 중 어느 쪽이 재현율을 더 크게 떨어뜨리나요? 데이터를 "많이" 모으는 것과 "제대로" 모으는 것의 차이를 한 문장으로 적어 보세요.